In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

In [2]:
#시그모이드함수
def sigmoid(z):
    return 1/(1+np.exp(-z))

In [3]:
def numerical_derivative(f, x): #수치미분함수
    delta_x = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=["multi_index"], op_flags=["readwrite"])

    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]

        x[idx] = float(tmp_val) + delta_x
        fx1 = f(x)                      # f(x + delta_x)

        x[idx] = float(tmp_val) - delta_x
        fx2 = f(x)                      # f(x - delta_x)

        grad[idx] = (fx1 - fx2) / (2 * delta_x)

        x[idx] = tmp_val
        it.iternext()

    return grad

In [5]:
class LogicGate:
    def __init__(self, gate_name, xdata, tdata):
        self.name = gate_name
        self.xdata = xdata.reshape(4, 2)  # 학습데이터
        self.tdata = tdata.reshape(4, 1)

        self.W = np.random.rand(self.xdata.shape[1], 1)  # 가중치
        self.b = np.random.rand(1)  # 바이어스
        self.learning_rate = 1e-2

    def loss_func(self):
        delta = 1e-7
        z = np.dot(self.xdata, self.W) + self.b
        y = sigmoid(z)
        # cross-entropy
        return -np.sum(self.tdata * np.log(y + delta) + (1 - self.tdata) * np.log((1 - y) + delta))

    def train(self):
        f = lambda x: self.loss_func()
        print("Initial loss value=", self.loss_func())
        for step in range(8001):
            self.W -= self.learning_rate * numerical_derivative(f, self.W)
            self.b -= self.learning_rate * numerical_derivative(f, self.b)
            if step % 1000 == 0:
                print("step=", step, "loss value=", self.loss_func())

    def predict(self, input_data):
        z = np.dot(input_data, self.W) + self.b
        y = sigmoid(z)
        if y > 0.5:
            result = 1
        else:
            result = 0
        return y, result

    def accuracy(self, test_xdata, test_tdata):
        match_list = []
        not_matched_list = []
        for index in range(len(test_xdata)):
            (real_val, logical_val) = self.predict(test_xdata[index])
            if logical_val == test_tdata[index]:
                match_list.append(index)
            else:
                not_matched_list.append(index)
        accuracy_val = len(match_list) / len(test_xdata)
        return accuracy_val

In [7]:
# AND 논리 게이트 검증
xdata = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
tdata = np.array([0, 0, 0, 1])
AND_obj = LogicGate("AND_GATE", xdata, tdata)
AND_obj.train()

Initial loss value= 5.026742967340558
step= 0 loss value= 4.961213974585372
step= 1000 loss value= 1.0037791831081324
step= 2000 loss value= 0.6589251303698473
step= 3000 loss value= 0.4906653627135121
step= 4000 loss value= 0.38979514046052965
step= 5000 loss value= 0.322575393337106
step= 6000 loss value= 0.27466197625699845
step= 7000 loss value= 0.23884830071091853
step= 8000 loss value= 0.21110768444561187


In [8]:
test_xdata = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = AND_obj.predict(input_data)
    print(input_data, "=", logical_val)

print("====================================")
test_tdata = np.array([0, 0, 0, 1])
accuracy_ret = AND_obj.accuracy(test_xdata, test_tdata)
print("Accuracy =>", accuracy_ret)

[0 0] = 0
[0 1] = 0
[1 0] = 0
[1 1] = 1
Accuracy => 1.0


In [11]:
# OR 논리 게이트 검증
xdata = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
tdata = np.array([0, 1, 1, 1])
OR_obj = LogicGate("OR_GATE", xdata, tdata)
OR_obj.train()

test_xdata = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = OR_obj.predict(input_data)
    print(input_data, "=", logical_val)

print("====================================")
test_tdata = np.array([0, 1, 1, 1])
accuracy_ret = OR_obj.accuracy(test_xdata, test_tdata)
print("Accuracy =>", accuracy_ret)

Initial loss value= 1.9183086084491994
step= 0 loss value= 1.910146827209029
step= 1000 loss value= 0.6933530905246735
step= 2000 loss value= 0.4214753604466595
step= 3000 loss value= 0.2984310964022729
step= 4000 loss value= 0.22943234885234984
step= 5000 loss value= 0.18567848723973437
step= 6000 loss value= 0.1556127019411424
step= 7000 loss value= 0.13374986344328554
step= 8000 loss value= 0.11717022878244683
[0 0] = 0
[0 1] = 1
[1 0] = 1
[1 1] = 1
Accuracy => 1.0


In [12]:
# NAND 논리 게이트 검증
xdata = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
tdata = np.array([1, 1, 1, 0])
NAND_obj = LogicGate("NAND_GATE", xdata, tdata)
NAND_obj.train()

Initial loss value= 2.8897804556299462
step= 0 loss value= 2.879653257947835
step= 1000 loss value= 1.0314758104955069
step= 2000 loss value= 0.6702345127798113
step= 3000 loss value= 0.49696648195469084
step= 4000 loss value= 0.39381827506139033
step= 5000 loss value= 0.3253610151605021
step= 6000 loss value= 0.2767003418602808
step= 7000 loss value= 0.24040159867559482
step= 8000 loss value= 0.21232883648728584


In [13]:
test_xdata = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = NAND_obj.predict(input_data)
    print(input_data, "=", logical_val)

print("====================================")
test_tdata = np.array([1, 1, 1, 0])
accuracy_ret = NAND_obj.accuracy(test_xdata, test_tdata)
print("Accuracy =>", accuracy_ret)

[0 0] = 1
[0 1] = 1
[1 0] = 1
[1 1] = 0
Accuracy => 1.0


In [19]:
# XOR 논리 게이트 검증
xdata = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
tdata = np.array([0, 1, 1, 0])
XOR_obj = LogicGate("XOR_GATE", xdata, tdata)
XOR_obj.train()

Initial loss value= 4.057384760153428
step= 0 loss value= 4.029678026893398
step= 1000 loss value= 2.772976818773225
step= 2000 loss value= 2.772603167488896
step= 3000 loss value= 2.7725885649045723
step= 4000 loss value= 2.772587949660926
step= 5000 loss value= 2.7725879234120963
step= 6000 loss value= 2.7725879222899885
step= 7000 loss value= 2.772587922242005
step= 8000 loss value= 2.772587922239953


In [20]:
test_xdata = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = XOR_obj.predict(input_data)
    print(input_data, "=", logical_val)

print("====================================")
test_tdata = np.array([0, 1, 1, 0])
accuracy_ret = XOR_obj.accuracy(test_xdata, test_tdata)
print("Accuracy =>", accuracy_ret)

[0 0] = 0
[0 1] = 0
[1 0] = 0
[1 1] = 1
Accuracy => 0.25


In [ ]:
# XOR = AND( NAND, OR )
test_xdata = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
test_tdata = np.array([0, 1, 1, 0])

for index, input_data in enumerate(test_xdata):
    (s1_sigmoid, s1) = NAND_obj.predict(input_data)   
    (s2_sigmoid, s2) = OR_obj.predict(input_data)     
    #s1(NAND)=[1,1,1,0]
    #s2(OR)=[0,1,1,1]
    new_input_data = np.array([s1, s2])
    #[1,0],[1,1],[1,1],[0,1]
    (sigmoid_val, logical_val) = AND_obj.predict(new_input_data)  
    print(input_data, "=", logical_val)


[0 0] = 0
[0 1] = 1
[1 0] = 1
[1 1] = 0


In [ ]:
# XOR = AND( NAND, OR )
test_xdata = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
test_tdata = np.array([0, 1, 1, 0])
NAND_XOR_LIST = []
for index, input_data in enumerate(test_xdata):
    (s1_sigmoid, s1) = NAND_obj.predict(input_data)
    (s2_sigmoid, s2) = OR_obj.predict(input_data)
    #s1(NAND)=[1,1,1,0]
    #s2(OR)=[0,1,1,1]
    new_input_data = np.array([s1, s2])
    #[1,0] -> [1,1] -> [1,1] -> [0,1]
    NAND_XOR_LIST.append(new_input_data)
    #[1,0],[1,1],[1,1],[0,1]

new_xdata = np.array(NAND_XOR_LIST)
XOR2_obj = LogicGate("XOR_2LAYER", new_xdata, test_tdata)
XOR2_obj.train()

for index, input_data in enumerate(test_xdata):
    (sigmoid_val, logical_val) = XOR2_obj.predict(new_xdata[index])
    print(input_data, "=", logical_val)

print("====================================")
accuracy_ret = XOR2_obj.accuracy(new_xdata, test_tdata)
print("Accuracy =>", accuracy_ret)

Initial loss value= 2.6460533150151644
step= 0 loss value= 2.642067710537795
step= 1000 loss value= 1.3084000195379242
step= 2000 loss value= 0.820139278401752
step= 3000 loss value= 0.5833614614840186
step= 4000 loss value= 0.44809090478548597
step= 5000 loss value= 0.36187937129798786
step= 6000 loss value= 0.30260589021350337
step= 7000 loss value= 0.2595506073482238
step= 8000 loss value= 0.22695382520732
[0 0] = 0
[0 1] = 1
[1 0] = 1
[1 1] = 0
Accuracy => 1.0


In [30]:
#딥러닝비스므리하게 XOR문제해결
#시그노이드 수치미분함수는 그대로 씀
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

#시그모이드함수
def sigmoid(z):
    return 1/(1+np.exp(-z))

def numerical_derivative(f, x): #수치미분함수
    delta_x = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=["multi_index"], op_flags=["readwrite"])

    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]

        x[idx] = float(tmp_val) + delta_x
        fx1 = f(x)                      # f(x + delta_x)

        x[idx] = float(tmp_val) - delta_x
        fx2 = f(x)                      # f(x - delta_x)

        grad[idx] = (fx1 - fx2) / (2 * delta_x)

        x[idx] = tmp_val
        it.iternext()

    return grad

In [37]:
class LogicGate:
    def __init__(self, gate_name, xdata, tdata):      # ydata → tdata, gata → gate
        self.name = gate_name
        self.xdata = xdata.reshape(4, 2)
        self.tdata = tdata.reshape(4, 1)

        # 은닉층: 입력 2개 → 노드 6개
        self.W2 = np.random.rand(2, 6)
        self.b2 = np.random.rand(6)

        # 출력층: 노드 6개 → 출력 1개
        self.W3 = np.random.rand(6, 1)
        self.b3 = np.random.rand(1)

        self.learning_rate = 1e-2                     # 추가 (train에서 사용)

    def loss_val(self):
        delta = 1e-7
        z2 = np.dot(self.xdata, self.W2) + self.b2
        a2 = sigmoid(z2)                              # sigmold → sigmoid
        z3 = np.dot(a2, self.W3) + self.b3
        y = a3 = sigmoid(z3)
        return -np.sum(self.tdata * np.log(y + delta) + (1 - self.tdata) * np.log((1 - y) + delta))
        #                                              ↑ 1+ 가 아니라 1-

    def train(self):
        f = lambda x: self.loss_val()                 # feed_forward → loss_val
        print("Initial loss value =", self.loss_val())
        for step in range(10001):
            self.W2 -= self.learning_rate * numerical_derivative(f, self.W2)
            self.b2 -= self.learning_rate * numerical_derivative(f, self.b2)
            self.W3 -= self.learning_rate * numerical_derivative(f, self.W3)
            self.b3 -= self.learning_rate * numerical_derivative(f, self.b3)
            if step % 1000 == 0:
                print("step =", step, "loss value =", self.loss_val())

    def predict(self, input_data):
        # self.xdata = input_data  ← 지움 (학습 데이터를 덮어써 버림)
        z2 = np.dot(input_data, self.W2) + self.b2
        a2 = sigmoid(z2)
        z3 = np.dot(a2, self.W3) + self.b3
        y = sigmoid(z3)
        if y > 0.5:
            result = 1
        else:
            result = 0
        return y, result

    def accuracy(self, test_xdata, test_tdata):
        match = 0
        for index in range(len(test_xdata)):
            (real_val, logical_val) = self.predict(test_xdata[index])
            if logical_val == test_tdata[index]:
                match += 1
        return match / len(test_xdata)

In [38]:
xdata = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
tdata = np.array([0, 1, 1, 0])
XOR_obj = LogicGate("XOR", xdata, tdata)
XOR_obj.train()

Initial loss value = 4.75593127662106
step = 0 loss value = 4.653149836483172
step = 1000 loss value = 2.765837398977589
step = 2000 loss value = 2.757147564113744
step = 3000 loss value = 2.7375123519164326
step = 4000 loss value = 2.690022504771227
step = 5000 loss value = 2.5821411424612153
step = 6000 loss value = 2.3715062143988526
step = 7000 loss value = 2.0420206133814207
step = 8000 loss value = 1.5951487586584552
step = 9000 loss value = 1.105962472604366
step = 10000 loss value = 0.737852784143716


In [39]:
test_xdata = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
for data in test_xdata:
    sigmoid_val, logical_val = XOR_obj.predict(data)
    print(data, "=", logical_val)

[0 0] = 0
[0 1] = 1
[1 0] = 1
[1 1] = 0


In [ ]:
#은닉층의 노트갯수줄이기 
class LogicGate:
    def __init__(self, gate_name, xdata, tdata):      # ydata → tdata, gata → gate
        self.name = gate_name
        self.xdata = xdata.reshape(4, 2)
        self.tdata = tdata.reshape(4, 1)

        self.W2 = np.random.rand(2, 2)
        self.b2 = np.random.rand(2)

        self.W3 = np.random.rand(2, 1)
        self.b3 = np.random.rand(1)

        self.learning_rate = 1e-2                     # 추가 (train에서 사용)

    def loss_val(self):
        delta = 1e-7
        z2 = np.dot(self.xdata, self.W2) + self.b2
        a2 = sigmoid(z2)                              # sigmold → sigmoid
        z3 = np.dot(a2, self.W3) + self.b3
        y = a3 = sigmoid(z3)
        return -np.sum(self.tdata * np.log(y + delta) + (1 - self.tdata) * np.log((1 - y) + delta))
        #                                              ↑ 1+ 가 아니라 1-

    def train(self):
        f = lambda x: self.loss_val()                 # feed_forward → loss_val
        print("Initial loss value =", self.loss_val())
        for step in range(12001):
            self.W2 -= self.learning_rate * numerical_derivative(f, self.W2)
            self.b2 -= self.learning_rate * numerical_derivative(f, self.b2)
            self.W3 -= self.learning_rate * numerical_derivative(f, self.W3)
            self.b3 -= self.learning_rate * numerical_derivative(f, self.b3)
            if step % 1000 == 0:
                print("step =", step, "loss value =", self.loss_val())

    def predict(self, input_data):
        # self.xdata = input_data  ← 지움 (학습 데이터를 덮어써 버림)
        z2 = np.dot(input_data, self.W2) + self.b2
        a2 = sigmoid(z2)
        z3 = np.dot(a2, self.W3) + self.b3
        y = sigmoid(z3)
        if y > 0.5:
            result = 1
        else:
            result = 0
        return y, result

    def accuracy(self, test_xdata, test_tdata):
        match = 0
        for index in range(len(test_xdata)):
            (real_val, logical_val) = self.predict(test_xdata[index])
            if logical_val == test_tdata[index]:
                match += 1
        return match / len(test_xdata)

In [47]:
xdata = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
tdata = np.array([0, 1, 1, 0])
XOR_obj = LogicGate("XOR", xdata, tdata)
XOR_obj.train()

Initial loss value = 4.103638823471018
step = 0 loss value = 4.06263116874309
step = 1000 loss value = 2.7671335042814373
step = 2000 loss value = 2.7616159718987445
step = 3000 loss value = 2.750556641433202
step = 4000 loss value = 2.726520063011105
step = 5000 loss value = 2.6721591717485462
step = 6000 loss value = 2.5560465184286048
step = 7000 loss value = 2.3695823364213813
step = 8000 loss value = 2.1617284072429244
step = 9000 loss value = 1.9418131109612893
step = 10000 loss value = 1.5913310834827201
step = 11000 loss value = 1.0827381213147507
step = 12000 loss value = 0.6855649622305071


In [48]:
test_xdata = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
for data in test_xdata:
    sigmoid_val, logical_val = XOR_obj.predict(data)
    print(data, "=", logical_val)

[0 0] = 0
[0 1] = 1
[1 0] = 1
[1 1] = 0


In [ ]:
#노드갯수를 늘리고 학습늘려도 잘안되면 은닉층추가해라

In [ ]:
#입력2개 / 은닉층 1 ->4개 / 은닉층 2 ->2개 / 출력층 ->1개  
# 입력 2개 / 은닉층1 → 4개 / 은닉층2 → 2개 / 출력층 → 1개
class LogicGate:
    def __init__(self, gate_name, xdata, tdata):
        self.name = gate_name
        self.xdata = xdata.reshape(4, 2)
        self.tdata = tdata.reshape(4, 1)

        # 은닉층1: 입력 2개 → 노드 4개
        self.W2 = np.random.rand(2, 4)
        self.b2 = np.random.rand(4)

        # 은닉층2: 노드 4개 → 노드 2개
        self.W3 = np.random.rand(4, 2)
        self.b3 = np.random.rand(2)

        # 출력층: 노드 2개 → 출력 1개
        self.W4 = np.random.rand(2, 1)
        self.b4 = np.random.rand(1)

        self.learning_rate = 1e-2     
    def loss_val(self):
        delta = 1e-7
        z2 = np.dot(self.xdata, self.W2) + self.b2
        a2 = sigmoid(z2)                       # 은닉층1 출력 (4,4)
        z3 = np.dot(a2, self.W3) + self.b3
        a3 = sigmoid(z3)                       # 은닉층2 출력 (4,2)
        z4 = np.dot(a3, self.W4) + self.b4
        y = a4 = sigmoid(z4)                   # 출력층 (4,1)
        return -np.sum(self.tdata * np.log(y + delta) + (1 - self.tdata) * np.log((1 - y) + delta))

    def train(self):
        f = lambda x: self.loss_val()
        print("Initial loss value =", self.loss_val())
        for step in range(12001):
            self.W2 -= self.learning_rate * numerical_derivative(f, self.W2)
            self.b2 -= self.learning_rate * numerical_derivative(f, self.b2)
            self.W3 -= self.learning_rate * numerical_derivative(f, self.W3)
            self.b3 -= self.learning_rate * numerical_derivative(f, self.b3)
            self.W4 -= self.learning_rate * numerical_derivative(f, self.W4)
            self.b4 -= self.learning_rate * numerical_derivative(f, self.b4)
            if step % 1000 == 0:
                print("step =", step, "loss value =", self.loss_val())

    def predict(self, input_data):
        z2 = np.dot(input_data, self.W2) + self.b2
        a2 = sigmoid(z2)
        z3 = np.dot(a2, self.W3) + self.b3
        a3 = sigmoid(z3)
        z4 = np.dot(a3, self.W4) + self.b4
        y = sigmoid(z4)
        if y > 0.5:
            result = 1
        else:
            result = 0
        return y, result

    def accuracy(self, test_xdata, test_tdata):
        match = 0
        for index in range(len(test_xdata)):
            (real_val, logical_val) = self.predict(test_xdata[index])
            if logical_val == test_tdata[index]:
                match += 1
        return match / len(test_xdata)
